# 10 · groupby & Pivot Tables

Splitting the table into groups and summarizing each one: groupby, agg with named results, and pivot_table.

In [1]:
# Setup: the steps from the previous parts (run this first)
import pandas as pd
pd.set_option("display.max_colwidth", 30)

URL = ("https://raw.githubusercontent.com/rfordatascience/tidytuesday/"
       "master/data/2021/2021-04-20/netflix_titles.csv")
df = pd.read_csv(URL)

df = df.drop(columns=["cast", "description"])

df["director"] = df["director"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")
df = df.dropna(subset=["date_added", "rating"])

df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()
df["duration_num"] = df["duration"].str.split(" ").str[0].astype(int)
df["main_country"] = df["country"].str.split(", ").str[0]

df["years_to_netflix"] = df["year_added"] - df["release_year"]
movies = df[df["type"] == "Movie"].assign(hours=pd.col("duration_num") / 60)

In [2]:
df.groupby("type")["duration_num"].mean()

type
Movie      99.330417
TV Show     1.761468
Name: duration_num, dtype: float64

In [3]:
df.groupby("year_added").size()

year_added
2008       2
2009       2
2010       1
2011      13
2012       3
2013      11
2014      25
2015      88
2016     440
2017    1222
2018    1684
2019    2153
2020    2009
2021     117
dtype: int64

In [4]:
by_country = df.groupby("main_country").agg(
    titles=("show_id", "count"),
    avg_year=("release_year", "mean"),
    newest=("release_year", "max"),
)
by_country.sort_values("titles", ascending=False).head()

,titles,avg_year,newest
main_country,,,
United States,2873,2013.249913,2021
India,956,2011.624477,2021
United Kingdom,575,2014.274783,2021
Unknown,505,2016.045545,2021
Canada,259,2016.142857,2021


In [5]:
df.pivot_table(index="year_added", columns="type", values="show_id", aggfunc="count").tail()

type,Movie,TV Show
year_added,,
2017,861.0,361.0
2018,1255.0,429.0
2019,1497.0,656.0
2020,1312.0,697.0
2021,88.0,29.0
